In [3]:
import sys
import os
import time
sys.path.append(os.path.abspath("../src"))
import pandas as pd
from data_loader import *

In [2]:
all_leagues = get_all_leagues()

In [3]:
tournaments = [
    "Slam IV",
    "FISSURE PLAYGROUND 2",
    "PGL Wallachia 2025 Season 6",
    "Slam V",
    "DreamLeague Season 27",
    "BLAST Slam VI",
    "DreamLeague Season 28",
]

In [4]:
leagues_ids = []

for name in tournaments:
    result = find_main_event_by_name(name, all_leagues)
    print(result)
    
    if not result.empty:
        leagues_ids.extend(result["leagueid"].tolist())
leagues_ids = list(set(leagues_ids))
print("Final League IDs:", leagues_ids)

      leagueid     name          tier
7751     17419  SLAM IV  professional
      leagueid                  name          tier
8946     18863  FISSURE PLAYGROUND 2  professional
      leagueid                         name          tier
8989     18920  PGL Wallachia 2025 Season 6  professional
      leagueid           name          tier
7752     17420         SLAM V  professional
9139     19099  BLAST SLAM VI  professional
      leagueid                   name          tier
9045     18988  DreamLeague Season 27  professional
      leagueid           name          tier
9139     19099  BLAST SLAM VI  professional
      leagueid                   name          tier
9279     19269  DreamLeague Season 28  professional
Final League IDs: [19269, 18920, 17419, 17420, 18988, 18863, 19099]


In [5]:
test_id = leagues_ids[0]
df = get_league_matches(test_id)
print(df.shape)

(195, 13)


In [6]:
all_matches = []

for lid in leagues_ids:
    df = get_league_matches(lid)
    print(f"League {lid}: {df.shape}")
    all_matches.append(df)
    time.sleep(1)
    
match_df = pd.concat(all_matches, ignore_index=True)
print("Total Matches:", match_df.shape)    

League 19269: (195, 13)
League 18920: (118, 13)
League 17419: (96, 13)
League 17420: (94, 13)
League 18988: (206, 13)
League 18863: (124, 13)
League 19099: (100, 13)
Total Matches: (933, 13)


In [7]:
match_df.to_csv("../data/raw/raw_matches.csv", index=False)

In [6]:
df = pd.read_csv("../data/processed/clean_matches.csv")

In [7]:
teams =  pd.concat([
    df["radiant_team_id"],
    df["dire_team_id"]
]).dropna().unique()

print(len(teams))

35


In [8]:
team_map = {}

for team_id in teams:
    data = get_team_info(int(team_id))
    team_map[team_id] = data.get("name")

    time.sleep(1)

In [11]:
df["radiant_team_name"] = df["radiant_team_id"].map(team_map)
df["dire_team_name"] = df["dire_team_id"].map(team_map)

In [12]:
team_df = pd.DataFrame(
    team_map.items(),
    columns=["team_id", "team_name"]
)

team_df.head()

,team_id,team_name
0,8291895,Tundra Esports
1,9467224,Aurora Gaming
2,2163,Team Liquid
3,8261500,Xtreme Gaming
4,9338413,MOUZ


In [10]:
team_df.to_csv("../data/processed/team_mapping.csv", index=False)